# ASR Evaluation Visualizations

Charts generated from evaluation results in `data/results/`. Run all cells to regenerate the 7 figures saved to `report/figures/`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['font.size'] = 11
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

# Color palette
SARVAM_COLOR = '#2563EB'   # blue
ELEVEN_COLOR = '#7C3AED'   # purple
WHISPER_COLOR = '#059669'  # green
ACCENT = '#DC2626'         # red for highlights

providers = ['Sarvam\nsaaras:v3', 'ElevenLabs\nscribe_v2', 'OpenAI\ngpt-4o-transcribe']
provider_colors = [SARVAM_COLOR, ELEVEN_COLOR, WHISPER_COLOR]

print("Setup complete.")

## 1. WER by Language (Latest Models)
The core result: Sarvam leads on Indian languages, Whisper wins Hinglish code-mixed.

In [ ]:
languages = ['Hindi\n(n=7)', 'Indian English\n(n=7)', 'Kannada\n(n=7)', 'Hinglish\n(n=4)', 'Kannada-English\n(n=3)']

sarvam_wer =  [9.10, 4.07, 27.31, 14.58, 27.38]
eleven_wer =  [14.47, 9.06, 48.78, 41.78, 76.39]
whisper_wer = [18.39, 13.39, 58.37, 9.05, 35.87]

x = np.arange(len(languages))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width, sarvam_wer, width, label='Sarvam saaras:v3', color=SARVAM_COLOR)
bars2 = ax.bar(x, eleven_wer, width, label='ElevenLabs scribe_v2 (raw)', color=ELEVEN_COLOR)
bars3 = ax.bar(x + width, whisper_wer, width, label='OpenAI gpt-4o-transcribe', color=WHISPER_COLOR)

ax.set_ylabel('Word Error Rate (%)')
ax.set_title('WER by Language — Latest Models (Lower is Better)', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(languages)
ax.legend(loc='upper left')
ax.set_ylim(0, 85)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8)

# Highlight Whisper's Hinglish win
ax.annotate('Best on Hinglish!', xy=(3 + width, 9.05), xytext=(3 + width + 0.3, 20),
            arrowprops=dict(arrowstyle='->', color=ACCENT), fontsize=9, color=ACCENT, fontweight='bold')

plt.tight_layout()
plt.savefig('../report/figures/fig1_wer_by_language.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")

## 2. The Script Mismatch Discovery — Three-Layer Code-Mixed WER
Raw WER, script-normalized WER, and entity accuracy on code-mixed banking recordings (n=7).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left panel: Raw vs Script-Normalized WER
prov_labels = ['Sarvam\nsaaras:v3', 'ElevenLabs\nscribe_v2', 'OpenAI\ngpt-4o-transcribe']
raw_wer = [20.06, 56.61, 20.55]
norm_wer = [20.06, 17.01, 19.90]

x = np.arange(len(prov_labels))
width = 0.35

bars_raw = ax1.bar(x - width/2, raw_wer, width, label='Raw WER', color=['#93C5FD', '#C4B5FD', '#6EE7B7'], edgecolor='gray')
bars_norm = ax1.bar(x + width/2, norm_wer, width, label='Script-Normalized WER', color=provider_colors, edgecolor='gray')

ax1.set_ylabel('WER (%)')
ax1.set_title('Code-Mixed WER: Raw vs Script-Normalized', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(prov_labels)
ax1.legend()
ax1.set_ylim(0, 65)

# Annotate the massive delta
ax1.annotate('-39.6pp!', xy=(1, 56.61), xytext=(1.4, 58),
            fontsize=11, color=ACCENT, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=ACCENT))

for bars in [bars_raw, bars_norm]:
    for bar in bars:
        h = bar.get_height()
        ax1.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

# Right panel: Entity Accuracy
entity_acc = [97.6, 100.0, 82.9]
bars_ent = ax2.bar(x, entity_acc, 0.5, color=provider_colors, edgecolor='gray')
ax2.set_ylabel('Entity Accuracy (%)')
ax2.set_title('Banking Entity Accuracy (41 entities)', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(prov_labels)
ax2.set_ylim(70, 105)
ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.3)

for bar, acc in zip(bars_ent, entity_acc):
    ax2.annotate(f'{acc:.1f}%', xy=(bar.get_x() + bar.get_width()/2, acc),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../report/figures/fig2_script_normalization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved.")

## 3. The 2x2 Correction Matrix
Model quality vs post-processing: correction benefit is inversely proportional to base model quality.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# Data: (base_wer, correction_delta, label, color, marker)
points = [
    (15.67, +0.76, 'Sarvam saarika:v2.5', SARVAM_COLOR, 's'),
    (15.14, +0.92, 'Sarvam saaras:v3', SARVAM_COLOR, 'o'),
    (25.24, -5.10, 'ElevenLabs scribe_v1', ELEVEN_COLOR, 's'),
    (32.23, -0.09, 'ElevenLabs scribe_v2', ELEVEN_COLOR, 'o'),
    (42.96, -15.28, 'Whisper whisper-1', WHISPER_COLOR, 's'),
    (27.68, +0.00, 'Whisper gpt-4o-transcribe', WHISPER_COLOR, 'o'),
]

for base, delta, label, color, marker in points:
    is_legacy = marker == 's'
    ax.scatter(base, delta, c=color, s=200, marker=marker, zorder=5,
              edgecolors='black', linewidth=1, alpha=0.9 if not is_legacy else 0.5)
    offset_x = 1.5 if 'Whisper' not in label else -1.5
    offset_y = 1.0
    ha = 'left' if 'Whisper' not in label else 'right'
    ax.annotate(label, (base, delta), textcoords="offset points",
               xytext=(offset_x*8, offset_y*8), fontsize=8, ha=ha,
               fontstyle='italic' if is_legacy else 'normal')

# Draw arrows from legacy to latest for each provider
arrow_pairs = [(0, 1), (2, 3), (4, 5)]
for i, j in arrow_pairs:
    ax.annotate('', xy=(points[j][0], points[j][1]),
               xytext=(points[i][0], points[i][1]),
               arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, alpha=0.5))

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Base Model WER (%)', fontsize=12)
ax.set_ylabel('WER Delta After LLM Correction (pp)', fontsize=12)
ax.set_title('The 2x2 Correction Matrix: Model Quality vs Post-Processing', fontweight='bold', fontsize=14)

# Add quadrant labels
ax.text(12, -10, 'Good model,\ncorrection helps\n(impossible zone)', fontsize=9, alpha=0.3, ha='center')
ax.text(40, -10, 'Weak model,\ncorrection helps', fontsize=9, alpha=0.3, ha='center')
ax.text(12, 2, 'Good model,\ncorrection unnecessary', fontsize=9, alpha=0.3, ha='center')
ax.text(40, 2, 'Weak model,\ncorrection harmful\n(overcorrection)', fontsize=9, alpha=0.3, ha='center')

# Legend for shapes
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='s', color='w', markerfacecolor='gray', markersize=10, label='Legacy model', alpha=0.5),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='Latest model'),
]
ax.legend(handles=legend_elements, loc='lower left')

plt.tight_layout()
plt.savefig('../report/figures/fig3_correction_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved.")

## 4. Model Evolution: Legacy vs Latest
How each provider improved (or didn't) between model generations.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: WER comparison
prov_short = ['Sarvam', 'ElevenLabs', 'OpenAI']
legacy_wer = [15.67, 25.24, 42.96]
latest_wer = [15.14, 32.23, 27.68]

x = np.arange(len(prov_short))
width = 0.35

bars_leg = ax1.bar(x - width/2, legacy_wer, width, label='Legacy model', color=['#93C5FD', '#C4B5FD', '#6EE7B7'], edgecolor='gray')
bars_lat = ax1.bar(x + width/2, latest_wer, width, label='Latest model', color=provider_colors, edgecolor='gray')

ax1.set_ylabel('Mean WER (%)')
ax1.set_title('WER: Legacy vs Latest Models', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(prov_short)
ax1.legend()

for bars in [bars_leg, bars_lat]:
    for bar in bars:
        h = bar.get_height()
        ax1.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

# Add delta annotations
deltas = [-0.53, +7.00, -15.29]
for i, d in enumerate(deltas):
    color = '#059669' if d < 0 else ACCENT
    sign = '+' if d > 0 else ''
    ax1.annotate(f'{sign}{d:.1f}pp', xy=(i, max(legacy_wer[i], latest_wer[i]) + 3),
                fontsize=10, fontweight='bold', ha='center', color=color)

ax1.set_ylim(0, 55)

# Right: Latency comparison
legacy_lat = [0.78, 2.73, 3.16]
latest_lat = [0.84, 1.14, 1.86]

bars_leg2 = ax2.bar(x - width/2, legacy_lat, width, label='Legacy model', color=['#93C5FD', '#C4B5FD', '#6EE7B7'], edgecolor='gray')
bars_lat2 = ax2.bar(x + width/2, latest_lat, width, label='Latest model', color=provider_colors, edgecolor='gray')

ax2.set_ylabel('Mean Latency (seconds)')
ax2.set_title('Latency: Legacy vs Latest Models', fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(prov_short)
ax2.legend()

for bars in [bars_leg2, bars_lat2]:
    for bar in bars:
        h = bar.get_height()
        ax2.annotate(f'{h:.2f}s', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

ax2.set_ylim(0, 4.0)

plt.tight_layout()
plt.savefig('../report/figures/fig4_model_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved.")

## 5. CER vs WER: Why WER Alone Is Misleading
For agglutinative Indian languages, CER tells a much better story than WER.

In [ ]:
# CER vs WER scatter — all providers, per language (legacy data from Day 3 with cleaner CER numbers)
fig, ax = plt.subplots(figsize=(10, 8))

# Data: (WER, CER, language, provider)
scatter_data = [
    # Sarvam
    (10.23, 2.70, 'hi', 'Sarvam'), (4.23, 2.80, 'en-IN', 'Sarvam'),
    (29.48, 11.09, 'ka', 'Sarvam'), (10.44, 2.64, 'hi-en', 'Sarvam'),
    (29.81, 7.52, 'kn-en', 'Sarvam'),
    # ElevenLabs
    (7.15, 2.74, 'hi', 'ElevenLabs'), (9.87, 6.21, 'en-IN', 'ElevenLabs'),
    (51.42, 18.16, 'ka', 'ElevenLabs'), (13.57, 3.71, 'hi-en', 'ElevenLabs'),
    (57.77, 40.36, 'kn-en', 'ElevenLabs'),
    # Whisper
    (26.69, 10.37, 'hi', 'Whisper'), (6.28, 3.76, 'en-IN', 'Whisper'),
    (89.76, 44.93, 'ka', 'Whisper'), (24.30, 10.14, 'hi-en', 'Whisper'),
    (82.20, 48.93, 'kn-en', 'Whisper'),
]

color_map = {'Sarvam': SARVAM_COLOR, 'ElevenLabs': ELEVEN_COLOR, 'Whisper': WHISPER_COLOR}
marker_map = {'hi': 'o', 'en-IN': 's', 'ka': '^', 'hi-en': 'D', 'kn-en': 'v'}

for wer, cer, lang, prov in scatter_data:
    ax.scatter(wer, cer, c=color_map[prov], s=120, marker=marker_map[lang],
              edgecolors='black', linewidth=0.5, zorder=5, alpha=0.8)

# Identity line
ax.plot([0, 100], [0, 100], 'k--', alpha=0.2, label='CER = WER')

# Label the Kannada cluster
ax.annotate('Kannada cluster:\nhigh WER, moderate CER\n= minor character errors,\nnot complete word failures',
           xy=(50, 20), fontsize=9, fontstyle='italic',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('WER (%)', fontsize=12)
ax.set_ylabel('CER (%)', fontsize=12)
ax.set_title('CER vs WER by Language and Provider (Legacy Models)', fontweight='bold', fontsize=14)

# Custom legend
from matplotlib.lines import Line2D
prov_handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=p)
               for p, c in color_map.items()]
lang_handles = [Line2D([0], [0], marker=m, color='w', markerfacecolor='gray', markersize=10, label=l)
               for l, m in marker_map.items()]
ax.legend(handles=prov_handles + lang_handles, loc='upper left', fontsize=9, ncol=2)

ax.set_xlim(0, 100)
ax.set_ylim(0, 55)

plt.tight_layout()
plt.savefig('../report/figures/fig5_cer_vs_wer.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved.")

## 6. Entity Accuracy: Legacy vs Latest
The metric that matters most for banking — did the model capture the right amounts, card types, and terms?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

prov_labels = ['Sarvam', 'ElevenLabs', 'OpenAI']
legacy_ent = [90.2, 70.7, 53.7]
latest_ent = [97.6, 100.0, 82.9]

x = np.arange(len(prov_labels))
width = 0.35

bars1 = ax.bar(x - width/2, legacy_ent, width, label='Legacy model', color=['#93C5FD', '#C4B5FD', '#6EE7B7'], edgecolor='gray')
bars2 = ax.bar(x + width/2, latest_ent, width, label='Latest model', color=provider_colors, edgecolor='gray')

ax.set_ylabel('Entity Accuracy (%)')
ax.set_title('Banking Entity Accuracy: Legacy vs Latest (41 entities, n=7 recordings)', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(prov_labels)
ax.legend()
ax.set_ylim(40, 108)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=10, fontweight='bold')

# Delta annotations
deltas_ent = [+7.4, +29.3, +29.2]
for i, d in enumerate(deltas_ent):
    ax.annotate(f'+{d:.1f}pp', xy=(i, max(legacy_ent[i], latest_ent[i]) + 4),
                fontsize=10, fontweight='bold', ha='center', color='#059669')

plt.tight_layout()
plt.savefig('../report/figures/fig6_entity_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 6 saved.")

## 7. Pricing vs Accuracy Tradeoff
At-scale monthly cost plotted against WER for a mid-size Indian bank (50K calls/day, 3 min avg).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# (monthly_cost_lakhs, wer, label, color)
pricing_points = [
    (22.5, 15.14, 'Sarvam saaras:v3\nRs 30/hr', SARVAM_COLOR),
    (17.6, 32.23, 'ElevenLabs scribe_v2 RT\n$0.28/hr (raw WER)', ELEVEN_COLOR),
    (22.7, 27.68, 'OpenAI gpt-4o-transcribe\n$0.36/hr', WHISPER_COLOR),
    (11.3, None, 'OpenAI gpt-4o-mini\n$0.18/hr (WER untested)', '#94A3B8'),
]

for cost, wer, label, color in pricing_points:
    if wer is not None:
        ax.scatter(cost, wer, c=color, s=300, zorder=5, edgecolors='black', linewidth=1)
        ax.annotate(label, (cost, wer), textcoords="offset points",
                   xytext=(12, -5), fontsize=9, ha='left')
    else:
        # Unknown WER — show as question mark marker
        ax.scatter(cost, 25, c=color, s=200, marker='$?$', zorder=5, alpha=0.5)
        ax.annotate(label, (cost, 25), textcoords="offset points",
                   xytext=(12, -5), fontsize=9, ha='left', alpha=0.6)

ax.set_xlabel('Monthly Cost (Rs Lakhs) — 50K calls/day, 3 min avg', fontsize=12)
ax.set_ylabel('Mean WER (%)', fontsize=12)
ax.set_title('Cost vs Accuracy at Scale (Latest Models)', fontweight='bold', fontsize=14)

# Ideal zone
ax.annotate('Ideal zone:\nlow cost, low WER', xy=(13, 12), fontsize=10,
           fontstyle='italic', alpha=0.4, ha='center',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.1))

ax.set_xlim(8, 28)
ax.set_ylim(8, 40)
ax.invert_yaxis()  # Lower WER is better, so invert

plt.tight_layout()
plt.savefig('../report/figures/fig7_pricing_vs_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 7 saved.")